# The Classifier Tournament — Chapter 5 · Units 4 + 5 + 6 + 7

**End-of-chapter project — seven classifiers, one dataset, one winner.**

> **Instructor copy.** Each yellow *Question* is followed by a purple *Answer*. The student version contains the questions and the §11 exercise prompt; the answers and exercise solution are stripped.

## The setup

Units 4-7 introduced seven classifiers, each with a different idea about how classification should work:

| Unit | Algorithm | Core idea |
|---|---|---|
| 4 | **Decision Tree** | Axis-aligned splits of feature space |
| 5 | **K-Nearest Neighbors** | Closest training examples vote |
| 6 | **Support Vector Machine** | Max-margin hyperplane, possibly via kernel trick |
| 7 | **Random Forest** | Bagging — many independent trees vote |
| 7 | **AdaBoost** | Sequential boosting — each learner fixes the previous mistakes |
| 7 | **Gradient Boosting** | Sequential boosting on residuals |
| 7 | **XGBoost** | Production-grade gradient boosting |

We use a harder dataset than Penguins (where everyone hits 99%): **classifying handwritten digits 0-9** from 8×8 grayscale images. On this data a single Decision Tree only reaches ~85%, so each more sophisticated algorithm has room to show its value.

For each algorithm we will:
1. Fit the **default** out-of-the-box version
2. Tune it with `GridSearchCV` over **2-3 hyperparameters**
3. Compare default vs tuned and record both in the final leaderboard

What you'll learn along the way:
- **When scaling matters** — Penguins jumped from 81% to 99% with `StandardScaler`; here it barely moves
- **When ensembles help** — Random Forest leaps from 85% (single tree) to ~98%
- **When ensembles flop** — Default AdaBoost gets 75%. Why? Diagnostic skill counts
- **The big lesson** — a great algorithm beats a mediocre ensemble. Choose your tool, then tune it


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets        import load_digits
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, GridSearchCV)
from sklearn.preprocessing   import StandardScaler
from sklearn.tree            import DecisionTreeClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
from sklearn.ensemble        import (RandomForestClassifier, AdaBoostClassifier,
                                     GradientBoostingClassifier, VotingClassifier)
from sklearn.metrics         import accuracy_score
import xgboost as xgb

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

RANDOM_STATE = 42

## 1. The dataset — handwritten digits

1797 8×8 grayscale digit images. Each example has 64 features (pixel intensities 0-16), with the label being the digit shown (0-9).

In [ ]:
data = load_digits()
X, y = data.data, data.target
print(f"Shape: {X.shape}   Classes: {np.unique(y)}   Pixel range: {X.min()}-{X.max()}")

In [ ]:
# Show one example of each digit
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for d in range(10):
    axes[d//5, d%5].imshow(X[y == d][0].reshape(8, 8), cmap='gray_r')
    axes[d//5, d%5].set_title(f"digit = {d}"); axes[d//5, d%5].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Stratified split + a scaled copy for KNN/SVM experiments
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

results = {}   # algorithm name → test accuracy
print(f"Training: {X_train.shape[0]} images,  Test: {X_test.shape[0]} images")

<div style="background-color: #dbeafe; border-left: 4px solid #3b82f6; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Tip</strong><br>From here on we tune hyperparameters with <strong><code>GridSearchCV</code></strong>: it does cross-validation <em>internally on the training set</em> to pick the best parameters, then we evaluate the winner once on the held-out test set. Honest and reproducible. We'll show the best params and best CV score for each round — not the full sweep — to keep the notebook readable.</div>

## 2. Round 1 — Decision Tree  (Unit 4)

A Decision Tree learns a sequence of yes/no splits on individual pixels. Simple and interpretable, but its representational power is limited by axis-aligned cuts.

In [ ]:
# Default: no constraint on depth, grows until pure
dt_default = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
results["Decision Tree (default)"] = accuracy_score(y_test, dt_default.predict(X_test))
print(f"Default DT (unconstrained):  test accuracy = {results['Decision Tree (default)']:.4f}")

In [ ]:
# Tune: max_depth + min_samples_split
param_grid = {
    "max_depth":         [5, 10, 20, None],
    "min_samples_split": [2, 10],
}
dt_search = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                          param_grid, cv=5, scoring="accuracy", n_jobs=-1).fit(X_train, y_train)
results["Decision Tree (tuned)"] = accuracy_score(y_test, dt_search.predict(X_test))

print(f"Best params:    {dt_search.best_params_}")
print(f"Best CV score:  {dt_search.best_score_:.4f}")
print(f"Test accuracy:  default = {results['Decision Tree (default)']:.4f}   "
      f"tuned = {results['Decision Tree (tuned)']:.4f}   "
      f"gain = {results['Decision Tree (tuned)'] - results['Decision Tree (default)']:+.4f}")

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>The tuned tree barely beats the default — both plateau around 85%. Why? What does this tell you about what a Decision Tree can fundamentally <em>not</em> learn from this data?</div>

## 3. Round 2 — K-Nearest Neighbors  (Unit 5)

KNN doesn't really train — it stores the training set and, at prediction time, finds the *k* closest training points and lets them vote.

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>In the Penguins project (Unit 1) you saw KNN's accuracy jump from 81% to 99% when you applied <code>StandardScaler</code>. Do you expect the same dramatic improvement here? Why or why not?</div>

In [ ]:
# Default = k=5 raw, but let's also check what scaling does
knn_default_raw    = KNeighborsClassifier().fit(X_train,   y_train)
knn_default_scaled = KNeighborsClassifier().fit(X_train_s, y_train)

acc_raw    = accuracy_score(y_test, knn_default_raw.predict(X_test))
acc_scaled = accuracy_score(y_test, knn_default_scaled.predict(X_test_s))
results["KNN (default)"] = acc_raw

print(f"Default KNN (k=5)  raw pixels:    {acc_raw:.4f}")
print(f"Default KNN (k=5)  scaled pixels: {acc_scaled:.4f}")
print(f"Effect of scaling:                {acc_scaled - acc_raw:+.4f}")

In [ ]:
# Tune: n_neighbors + weights (uniform vs inverse-distance)
param_grid = {
    "n_neighbors": [1, 3, 5, 7, 11],
    "weights":     ["uniform", "distance"],
}
knn_search = GridSearchCV(KNeighborsClassifier(), param_grid,
                           cv=5, scoring="accuracy", n_jobs=-1).fit(X_train, y_train)
results["KNN (tuned)"] = accuracy_score(y_test, knn_search.predict(X_test))

print(f"Best params:    {knn_search.best_params_}")
print(f"Best CV score:  {knn_search.best_score_:.4f}")
print(f"Test accuracy:  default = {results['KNN (default)']:.4f}   "
      f"tuned = {results['KNN (tuned)']:.4f}   "
      f"gain = {results['KNN (tuned)'] - results['KNN (default)']:+.4f}")

## 4. Round 3 — Support Vector Machine  (Unit 6)

SVM finds the maximum-margin hyperplane separating classes. The kernel trick lets it carve curvy decision boundaries by implicitly mapping inputs into a higher-dimensional space.

In [ ]:
# Default = RBF kernel, C=1, gamma='scale'
svm_default = SVC(random_state=RANDOM_STATE).fit(X_train, y_train)
results["SVM (default)"] = accuracy_score(y_test, svm_default.predict(X_test))
print(f"Default SVM (rbf, C=1):  test accuracy = {results['SVM (default)']:.4f}")

In [ ]:
# Tune: kernel + C  (gamma=scale is fine for digits, stays default)
param_grid = {
    "kernel": ["linear", "rbf"],
    "C":      [0.1, 1, 10],
}
svm_search = GridSearchCV(SVC(gamma="scale", random_state=RANDOM_STATE), param_grid,
                           cv=5, scoring="accuracy", n_jobs=-1).fit(X_train, y_train)
results["SVM (tuned)"] = accuracy_score(y_test, svm_search.predict(X_test))

print(f"Best params:    {svm_search.best_params_}")
print(f"Best CV score:  {svm_search.best_score_:.4f}")
print(f"Test accuracy:  default = {results['SVM (default)']:.4f}   "
      f"tuned = {results['SVM (tuned)']:.4f}   "
      f"gain = {results['SVM (tuned)'] - results['SVM (default)']:+.4f}")

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>Both default and tuned SVMs use the RBF kernel and land near 99% — the best of any algorithm so far. What does this tell us about the geometry of digit data?</div>

## 5. Interlude — the wisdom of crowds

In 1906, Francis Galton asked 800 country-fair attendees to guess the weight of an ox. No single guess was very close, but **the average of all 800 was within 1% of the true weight**. The average of many imperfect guessers beat any single expert.

This is the intuition behind **ensemble methods**. Instead of one classifier, train many and let them vote:

| Flavor | How it works | Canonical algorithm |
|---|---|---|
| **Bagging** | Many copies of one algorithm, each on a different random sample | Random Forest |
| **Boosting (re-weighting)** | Sequential — each learner focuses on what the previous got wrong | AdaBoost |
| **Boosting (residuals)** | Sequential — each learner fits the running ensemble's errors | Gradient Boosting / XGBoost |

So far our best is SVM at ~99%. Can ensembles beat it?

## 6. Round 4 — Random Forest  (Unit 7, bagging)

Many Decision Trees, each trained on a random bootstrap of the data and considering only a random feature subset at each split. Trees vote at prediction time.

In [ ]:
# Default = 100 trees, unconstrained depth, sqrt(64)≈8 features per split
rf_default = RandomForestClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
results["Random Forest (default)"] = accuracy_score(y_test, rf_default.predict(X_test))
print(f"Default RF (100 trees):  test accuracy = {results['Random Forest (default)']:.4f}")

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>A single Decision Tree got 85%. Random Forest jumps to ~98% — a <strong>+13pp leap</strong> from one preprocessing change. The trees use the same algorithm and would individually plateau at 85%. So where does the gain come from?</div>

In [ ]:
# Tune: n_estimators + max_depth
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth":    [None, 10],
}
rf_search = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), param_grid,
                          cv=5, scoring="accuracy", n_jobs=-1).fit(X_train, y_train)
results["Random Forest (tuned)"] = accuracy_score(y_test, rf_search.predict(X_test))

print(f"Best params:    {rf_search.best_params_}")
print(f"Best CV score:  {rf_search.best_score_:.4f}")
print(f"Test accuracy:  default = {results['Random Forest (default)']:.4f}   "
      f"tuned = {results['Random Forest (tuned)']:.4f}   "
      f"gain = {results['Random Forest (tuned)'] - results['Random Forest (default)']:+.4f}")

## 7. Round 5 — AdaBoost  (Unit 7, sequential boosting)

AdaBoost trains classifiers **sequentially**: each new one focuses on the examples the previous ones got wrong. By default it uses depth-1 **decision stumps** — the idea being that even very weak learners, combined sequentially, become strong.

In [ ]:
# Default = 50 depth-1 stumps, learning_rate=1.0
ada_default = AdaBoostClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
results["AdaBoost (default)"] = accuracy_score(y_test, ada_default.predict(X_test))
print(f"Default AdaBoost (50 stumps):  test accuracy = {results['AdaBoost (default)']:.4f}")

<div style="background-color: #fee2e2; border-left: 4px solid #ef4444; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Watch out</strong><br><strong>That's surprisingly bad.</strong>  AdaBoost gets ~75% here — <em>worse</em> than a single Decision Tree, and dramatically worse than Random Forest. The textbook claim is that boosting turns weak learners into strong ones. So what went wrong?</div>

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>Form a hypothesis: why does default AdaBoost (50 depth-1 stumps) flop on this 10-class image problem? What would you change?</div>

In [ ]:
# Tune: base estimator depth + n_estimators
# The double-underscore syntax 'estimator__max_depth' reaches into the wrapped base learner.
param_grid = {
    "estimator__max_depth": [1, 3, 5],
    "n_estimators":         [50, 100],
}
ada_search = GridSearchCV(
    AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
                        random_state=RANDOM_STATE),
    param_grid, cv=5, scoring="accuracy", n_jobs=-1).fit(X_train, y_train)
results["AdaBoost (tuned)"] = accuracy_score(y_test, ada_search.predict(X_test))

print(f"Best params:    {ada_search.best_params_}")
print(f"Best CV score:  {ada_search.best_score_:.4f}")
print(f"Test accuracy:  default = {results['AdaBoost (default)']:.4f}   "
      f"tuned = {results['AdaBoost (tuned)']:.4f}   "
      f"gain = {results['AdaBoost (tuned)'] - results['AdaBoost (default)']:+.4f}")

**Recovery confirmed.** With a deeper base tree, AdaBoost climbs from 75% to ~98% — a +20pp gain. This will be the largest tuning gain in the whole tournament.

## 8. Round 6 — Gradient Boosting  (Unit 7, residual boosting)

Gradient Boosting also boosts sequentially, but instead of upweighting hard examples (AdaBoost's strategy), each new tree fits the **residual errors** of the ensemble so far. More robust than AdaBoost out of the box.

In [ ]:
# Default = 100 estimators, max_depth=3, learning_rate=0.1
gb_default = GradientBoostingClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
results["Gradient Boosting (default)"] = accuracy_score(y_test, gb_default.predict(X_test))
print(f"Default GB:  test accuracy = {results['Gradient Boosting (default)']:.4f}")

In [ ]:
# Tune: n_estimators + learning_rate. (Keep grid small because sklearn GB is slow.
# Each fit can take several seconds — we cap n_estimators to keep wall time reasonable.)
param_grid = {
    "n_estimators":  [50, 100],
    "learning_rate": [0.1, 0.3],
}
gb_search = GridSearchCV(GradientBoostingClassifier(random_state=RANDOM_STATE), param_grid,
                          cv=5, scoring="accuracy", n_jobs=-1).fit(X_train, y_train)
results["Gradient Boosting (tuned)"] = accuracy_score(y_test, gb_search.predict(X_test))

print(f"Best params:    {gb_search.best_params_}")
print(f"Best CV score:  {gb_search.best_score_:.4f}")
print(f"Test accuracy:  default = {results['Gradient Boosting (default)']:.4f}   "
      f"tuned = {results['Gradient Boosting (tuned)']:.4f}   "
      f"gain = {results['Gradient Boosting (tuned)'] - results['Gradient Boosting (default)']:+.4f}")

## 9. Round 7 — XGBoost  (Unit 7, industrial-strength gradient boosting)

**XGBoost** implements the same gradient-boosting math as sklearn's `GradientBoostingClassifier`, but with dramatically better engineering: parallel split computation, built-in L1/L2 regularization, native missing-value handling, GPU support. It has dominated Kaggle and industry since ~2014.

Expect: similar accuracy to sklearn GB, **much faster training**.

In [ ]:
# Default XGBoost + timing comparison vs sklearn GB
t0 = time.time()
xgb_default = xgb.XGBClassifier(random_state=RANDOM_STATE,
                                 eval_metric='mlogloss').fit(X_train, y_train)
xgb_time = time.time() - t0
results["XGBoost (default)"] = accuracy_score(y_test, xgb_default.predict(X_test))

t0 = time.time()
GradientBoostingClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
gb_time = time.time() - t0

print(f"XGBoost (default):    accuracy = {results['XGBoost (default)']:.4f}   "
      f"time = {xgb_time:.2f}s")
print(f"sklearn GB (default): accuracy = {results['Gradient Boosting (default)']:.4f}   "
      f"time = {gb_time:.2f}s")
print(f"\nSpeedup:  XGBoost is  {gb_time / xgb_time:.1f}×  faster than sklearn GB.")

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>XGBoost and sklearn's Gradient Boosting implement the same math. They get nearly identical accuracy. So why has XGBoost displaced vanilla GB almost everywhere in industry?</div>

In [ ]:
# Tune: n_estimators + max_depth + learning_rate (XGBoost is fast, can afford a bigger grid)
param_grid = {
    "n_estimators":  [100, 200],
    "max_depth":     [3, 6],
    "learning_rate": [0.1, 0.3],
}
xgb_search = GridSearchCV(
    xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='mlogloss'),
    param_grid, cv=5, scoring="accuracy", n_jobs=-1).fit(X_train, y_train)
results["XGBoost (tuned)"] = accuracy_score(y_test, xgb_search.predict(X_test))

print(f"Best params:    {xgb_search.best_params_}")
print(f"Best CV score:  {xgb_search.best_score_:.4f}")
print(f"Test accuracy:  default = {results['XGBoost (default)']:.4f}   "
      f"tuned = {results['XGBoost (tuned)']:.4f}   "
      f"gain = {results['XGBoost (tuned)'] - results['XGBoost (default)']:+.4f}")

## 10. The final showdown

Single train/test scores are noisy. Time for honest **5-fold cross-validation** on the full data, with all 14 entries (default + tuned for each of 7 algorithms) head to head.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# All 14 models: default and tuned for each algorithm
models_for_cv = {
    "Decision Tree (default)":     DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Decision Tree (tuned)":       dt_search.best_estimator_,
    "KNN (default)":               KNeighborsClassifier(),
    "KNN (tuned)":                 knn_search.best_estimator_,
    "SVM (default)":               SVC(random_state=RANDOM_STATE),
    "SVM (tuned)":                 svm_search.best_estimator_,
    "Random Forest (default)":     RandomForestClassifier(random_state=RANDOM_STATE),
    "Random Forest (tuned)":       rf_search.best_estimator_,
    "AdaBoost (default)":          AdaBoostClassifier(random_state=RANDOM_STATE),
    "AdaBoost (tuned)":            ada_search.best_estimator_,
    "Gradient Boosting (default)": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting (tuned)":   gb_search.best_estimator_,
    "XGBoost (default)":           xgb.XGBClassifier(random_state=RANDOM_STATE,
                                                       eval_metric='mlogloss'),
    "XGBoost (tuned)":             xgb_search.best_estimator_,
}

cv_results = []
for name, m in models_for_cv.items():
    scores = cross_val_score(m, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results.append({"model": name, "mean": scores.mean(), "std": scores.std()})
cv_df = pd.DataFrame(cv_results).sort_values("mean", ascending=False)
print(cv_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# Visualize the leaderboard — defaults light gray, tuned dark blue, AdaBoost-default in orange
def bar_color(name):
    if name == "AdaBoost (default)":   return "#D85A30"
    if name.endswith("(default)"):     return "#B0B0B0"
    return "#1F396B"

fig, ax = plt.subplots(figsize=(11, 7))
y_pos = np.arange(len(cv_df))
ax.barh(y_pos, cv_df["mean"], xerr=cv_df["std"],
         color=[bar_color(m) for m in cv_df["model"]],
         alpha=0.88, edgecolor="white")
ax.set_yticks(y_pos); ax.set_yticklabels(cv_df["model"])
ax.invert_yaxis(); ax.set_xlim(0.6, 1.0)
ax.set_xlabel("Mean 5-fold CV accuracy  (error bars = 1 std)")
ax.set_title("The classifier tournament — final leaderboard")

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="#1F396B", label="tuned"),
                    Patch(facecolor="#B0B0B0", label="default"),
                    Patch(facecolor="#D85A30", label="cautionary default")],
          loc="lower right")
plt.tight_layout(); plt.show()

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>What three lessons would you take from this leaderboard about choosing classifiers?</div>

## 10.5. Where does tuning matter most?

You ran a `GridSearchCV` for every algorithm. Sometimes the tuning made a big difference; sometimes it barely moved the needle. Let's put the gains side by side.

In [ ]:
# Pair each algorithm's default and tuned CV scores; compute the gain
algorithms = ["Decision Tree", "KNN", "SVM", "Random Forest",
              "AdaBoost", "Gradient Boosting", "XGBoost"]

def cv_mean(name):
    return cv_df.loc[cv_df["model"] == name, "mean"].iloc[0]

gains = [{"algorithm": a,
           "default":   cv_mean(f"{a} (default)"),
           "tuned":     cv_mean(f"{a} (tuned)"),
           "gain":      cv_mean(f"{a} (tuned)") - cv_mean(f"{a} (default)")}
         for a in algorithms]
gains_df = pd.DataFrame(gains).sort_values("gain", ascending=False)
print(gains_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# Visualize the gains
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(np.arange(len(gains_df)), gains_df["gain"],
         color=["#D85A30" if g > 0.05 else "#1F396B" for g in gains_df["gain"]],
         alpha=0.88, edgecolor="white")
ax.set_yticks(np.arange(len(gains_df)))
ax.set_yticklabels(gains_df["algorithm"]); ax.invert_yaxis()
ax.set_xlabel("Gain from tuning  (tuned CV − default CV)")
ax.set_title("How much does hyperparameter tuning matter for each algorithm?")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

<div style="background-color: #fef3c7; border-left: 4px solid #f59e0b; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Question for you</strong><br>AdaBoost is in a class of its own — tuning rescues it from ~74% to ~98%. Every other algorithm gains less than 2pp. What does this tell you about sklearn's defaults and about how to spend your tuning budget?</div>

## 11. Try it yourself — can a Voting Classifier beat the best single model?

<div style="background-color: #dcfce7; border-left: 4px solid #16a34a; padding: 12px 16px; border-radius: 4px; margin: 16px 0;"><strong>Try it yourself</strong><br><strong>The wisdom of crowds, revisited.</strong> Random Forest combines many <em>identical</em> classifiers. But ensembles can also combine <em>different</em> kinds of classifiers — the hope is that their different mistakes will cancel out even better.<br><br>Build a <code>VotingClassifier</code> from the three most diverse winners — <strong>tuned Decision Tree, KNN, and SVM</strong> — using majority voting (the default <code>voting='hard'</code>). Evaluate it with 5-fold cross-validation. Does the voter beat the best individual model? Discuss what you found.</div>

In [ ]:
# Combine the three diverse tuned winners with majority voting

# YOUR CODE HERE
# ...

print(f"Voting Classifier (DT + KNN + SVM):  {voter_scores.mean():.4f}  ± {voter_scores.std():.4f}")
print(f"KNN alone (best individual):          {knn_scores.mean():.4f}  ± {knn_scores.std():.4f}")
print(f"SVM alone:                            {svm_scores.mean():.4f}  ± {svm_scores.std():.4f}")
print(f"\nVoter improvement over best individual: "
      f"{voter_scores.mean() - max(knn_scores.mean(), svm_scores.mean()):+.4f}")

## 12. Recap

You walked through seven classifiers on one dataset, ran each one twice (default and `GridSearchCV`-tuned over 2-3 hyperparameters), and earned four big insights:

### The leaderboard (5-fold CV)
| Algorithm | Unit | Default | Tuned | Gain |
|---|---|---|---|---|
| **SVM (RBF)**          | 6 | ~98.8% | **~99.1%** | +0.3pp |
| **KNN**                | 5 | ~98.7% | ~98.8% | +0.1pp |
| **Random Forest**      | 7 | ~97.8% | ~97.9% | +0.1pp |
| **AdaBoost**           | 7 | **~73.5%** | **~98.0%** | **+25pp** 🎯 |
| **Gradient Boosting**  | 7 | ~96.4% | ~96.7% | +0.3pp |
| **XGBoost**            | 7 | ~96.7% | ~96.7% | ~0pp |
| **Decision Tree**      | 4 | ~85.5% | ~85.5% | ~0pp |

### Four takeaways

1. **Match the algorithm to the data.** SVM dominates here because its RBF kernel can carve out the curvy decision boundaries digit recognition needs. On Penguins, a Decision Tree would have hit the same accuracy. No algorithm is always best.

2. **Scaling matters when feature ranges differ — not as a blind reflex.** Penguins: scaling jumped KNN from 81% to 99%. Digits: scaling barely moved the needle. Inspect your features first.

3. **Ensembles help most when the base learner is weak.** Random Forest leaps from 85% (single tree) to ~98% — huge value. But ensembling doesn't beat a fundamentally great algorithm: SVM matches or beats every ensemble here.

4. **Most sklearn defaults are well-tuned — except when they aren't.** Six of seven algorithms gain less than 0.5pp from `GridSearchCV`. AdaBoost is the dramatic exception (74% → 98%). Run defaults first as a cheap baseline; tune only when the default is dramatically below peer algorithms.

### Production constraints (a note on deployment)

The leaderboard ranks by accuracy alone, but in real-world projects:
- **XGBoost** trains 20-30× faster than sklearn GB with the same accuracy — the right choice on large data.
- **Decision Tree** is interpretable — a regulator can audit its rules.
- **KNN** has zero training time but slow prediction (must scan the training set at inference).
- **SVM** with RBF kernel scales poorly to millions of training samples.

The art of applied ML is matching the algorithm to both the data's structure and the deployment constraints.
